# 2026/8/13
## Deepseek-R1论文精读

> DeepSeek-R1 的突破之一在于利用 GRPO 强化学习激发模型的推理能力。GRPO 通过对同一问题生成的一组回答进行相对奖励比较来估计优势，无需额外的 Critic/Value Model，因此能够降低 RL 阶段的计算与显存开销。

## Deepseek-R1-Zero 技术内容

### 1. GRPO (Group Relative Policy Optimization)

#### 核心思路

<div align="center">

<img src="note_pics/GRPO.png">

<br>

GRPO 通过让模型针对同一个问题生成多个回答，并根据这些回答的相对奖励来估计 Advantage，从而不需要额外训练一个独立的 Value/Critic 网络。由于大模型的 Value 网络本身需要大量参数、显存和计算资源，因此去掉它可以显著降低 RL 训练成本

<br>

<img src="note_pics/example.png">
<br>

举例

<br>
</div>


### 2. 基于规则奖励模型 (Rule-based reward)

开发 **DeepSeek-R1-Zero** 时使用的奖励模型

- 准确性奖励 (Accuracy rewards)

  准确性奖励模型⽤于评估模型的回答**是否正确(T/F)**

  - 如果是数学问题，模型需要给出⼀个确定的答案，并以指定的格式呈现（例如在⼀个框框内），这样就可以通过规则的⽅式验证答案是否正确

  - 像 LeetCode 这种编程题⽬，可以使⽤编译器来⾃动⽣成反馈，判断模型提供的答案是否通过了预设的测试⽤例，从⽽判断答案的准确性

- 格式奖励 (Format rewards) 

  除了准确性奖励模型之外，还使⽤了格式奖励模型，这个模型的作⽤是要求模型将其思考过程放在 <think> 和 </think> 标签之间。也就是说，模型不仅要给出最终的答案，还需要明确地表达其推理过程，这对于训练推理能⼒和提升模型的透明度⾮常重要。

$$
Reward_{rule} = Reward_{acc} + Reward_{format}
$$

>问：为什么不使⽤基于神经⽹络的奖励模型？

研究发现，在⼤规模的强化学习过程中，使⽤神经奖励模型可能会导致 **奖励劫持 (reward hacking) ** 的问题。

奖励劫持是指模型在训练时可能通过⼀些不被预期的⽅式获得奖励，从⽽偏离了真实⽬标或产⽣了不符合实际需求的⾏为。

- 例：我要训练模型回答更加详细，因此奖励模型根据回答精细度的提高而给出更高分数。然后模型可能会发现为了取得更高分数我需要详细回答每个问题，导致对于一个很简单的问题，它都会这样啰嗦地回答

  ```回答：首先，我们需要理解这个问题...其次，我们需要分析这个问题...进一步来说，我们还需要考虑...综上所述...因此...总而言之...```

  > 用户说一个 hello，它都想了很多才进行回答（有些人说R1是一个很“内耗”的模型也是有这个原因）原文：*Nevertheless, there remains room for further optimization in terms of token efficiency, as instances of excessive reasoning—manifested as overthinking—are still observed in response to simpler questions.*

因此，**基于规则的奖励**模型相较于神经奖励模型，能够**更加可靠和稳定地引导模型**朝着正确的⽅向进⾏优化。


### 3. 训练模板

这种设计的⽬的是让模型⾃然地展示其推理过程和问题解决⽅式，⽽**不是通过强加特定的推理规则或策略来引导模型⾏为**。通过这种⽅式，研究⼈员能够准确地观察到模型在强化学习过程中的⾃然进展，从⽽为训练提供更加真实的数据。

> 仅仅使用这种简单的提示词，在强化学习过程中模型就能产生出**自我反思**的能力 (aha moment)

<div align="center">

<img src="note_pics/R1-zero-template.png">

</div>

- 为了训练 DeepSeek-R1-Zero，⾸先设计了⼀个简单的模板，⽤于引导模型遵循特定的指令。在该模板中，模型的输出包括两个部分：

  - 推理过程：模型⾸先需要展示其推理过程，即在得出答案之前，必须详细地表达它是如何思考的，这部分可以帮助我们了解模型的推理路径。

  - 最终答案：在展示了推理过程后，模型给出最终的答案。这⼀结构化的输出⽅式有助于清晰地区分推理和最终结论，从⽽在强化学习过程中帮助更好地评估和优化模型。

- 在设计这个模板时，研究⼈员特意避免了对内容的过多限制。**模板的约束仅限于结构⽅⾯**，避免了引⼊任何特定的内容偏⻅。例如：

  - 不强制要求反思性推理，即模型不被强迫去做多余的推理反思； 

  - 不推动特定的解决策略，例如没有指定某些特定的数学或逻辑推理⽅法来解决问题。

### 4. 自我进化过程

通过从基础模型（仅经过预训练）直接开始强化学习，研究⼈员能够清楚地观察到模型如何在 **没有监督微调** 的影响下，逐步提升其能⼒，尤其是在处理复杂推理任务时的表现。

<div align="center">

<img src="note_pics/evolve.png">

</div>

如图所示，DeepSeek-R1-Zero 在训练过程 **平均思考时间** 随着训练的进⾏持续增加。这个提升是 **模型内部发展的⾃然结果** 。随着训练的推进，模型逐渐学会解决越来越复杂的推理任务，并通过延⻓测试阶段的计算时间来实现这⼀⽬标。这些计算过程可能涉及⽣成数百到数千个token，帮助模型更深⼊地探索和优化其思维过程。


### 5. Aha Moment

这个时刻出现在模型的中期版本中。在这个阶段，模型学会了通过重新评估最初的解决⽅法，分配更多的思考时间给⼀个问题。这⼀⾏为不仅体现了模型推理能⼒的提升，还展示了强化学习如何能够引导模型达到意想不到且复杂的结果。

<div align="center">

<img src="note_pics/aha.png" >

</div>

> 原文：*The self-evolution of DeepSeek-R1-Zero underscores the power and beauty of RL: rather than explicitly teaching the model how to solve a problem, we simply provide it with the right incentives, and it autonomously develops advanced problem-solving strategies.*

可以看出，在模型训练中，RL 与 SFT 是两个不同的学习路径。前者强调给予模型合适的奖励以让其**自己探索求解过程**，从而具备推理能力；后者强调给予模型充足的用于微调的数据集。

相比 SFT，**RL 对人工标注的逐条推理示范依赖较低**

- DeepSeek-R1-Zero 的局限性

  尽管 DeepSeek-R1-Zero 展示出了强⼤的推理能⼒，但它仍⾯临⼀些如可读性⽅⾯较差、语⾔混杂的现象。

## Deepseek-R1 带有冷启动的强化学习

在看到 DeepSeek-R1-Zero 取得的显著效果后，研究者提出了两个⾃然的问题：

1. 如果引⼊少量⾼质量的“冷启动”数据（先稍微告诉模型如何进行推理），是否能在推理性能上取得更⼤提升或加快收敛速度？

2. 如何训练出在保证强⼤推理能⼒的同时，⼜能输出易读、连贯的推理链（CoT），且具有更⼴泛通⽤能⼒的模型？

DeepSeek-R1 为了避免直接从基础模型进⾏强化学习时可能出现的“早期不稳定”问题，会在 强化学习之前 使⽤少量 “⻓推理链（long CoT）”的⾼质量数据对模型进⾏⼀次微调。

这样做的好处是，模型在进⼊⼤规模强化学习阶段前，已经**具备了初步的推理能⼒和输出格式**，从⽽可以加速收敛并提⾼最终效果。

<div align="center">

<img src="note_pics/R1-training_pipeline.png" >
<br>
Deepseek-R1 训练流程
</div>

### 1. 冷启动 (cold start)

> 在正式进行大规模 RL 之前，先给模型一小批高质量的“推理示范数据”，让模型先学会一个基本的推理格式和行为模式，这样就能保证训练数据能够与后续的大规模 RL 有更相似的分布。冷启动过程本质是一个 SFT，相当于开车前让教练带我开一会儿，而不是从头开始摸索如何开车。

本研究中，研究者收集了数千条“冷启动”数据来对 DeepSeek-V3-Base 进⾏初步微调，从⽽作为强化学习阶段的初始策略。

#### 数据收集

- **冷启动数据**来自多种途径，不是仅仅获取 R1-Zero 的输出：

  - Few-shot Prompting + 提示模型进行反思与验证
  
    提供少量包含高质量长链式推理的示例，引导模型生成类似的 reasoning data 并且 prompt 鼓励模型在解决问题时进行自我检查、反思和验证，从而生成更加完整的推理过程
  
  - 利用 DeepSeek-R1-Zero
  
    收集 R1-Zero 在 RL 探索过程中产生的 reasoning outputs，并对其中高质量的推理结果进行筛选和整理
  
  - 人工筛选与后处理
  
    由人工对生成的数据进行质量控制、错误筛除和格式整理，保证 reasoning data 的正确性、可读性和一致性。

#### 冷启动作用

冷启动数据中融⼊了⼈类先验，精⼼设计的格式和示例能够帮助模型以更好的起点进⼊强化学习训练。

### 2. ⾯向推理的强化学习

使⽤冷启动数据对 DeepSeek-V3-Base 做完微调后，研究者接着对其进⾏与 DeepSeek-R1-Zero 相同的⼤规模强化学习训练，此阶段的重点是进⼀步强化模型的推理能⼒

该步骤的奖励函数与 DeepSeek-R1-Zero 相似，但是加上了**语言一致奖励**以解决语言混用问题

$$
Reward_{rule} = Reward_{acc} + Reward_{format}
$$
$$
Reward_{language}=\frac{Num(Words_{target})}{Num(Words)}
$$

### 3. 拒绝采样与监督微调 (Rejection Sampling & Supervised Fine-Tuning)

经过上一步的强化学习，模型是一个会解题的理科生，没有通用的模型能力比如写作、⻆⾊扮演、常识问答等，因此需要扩充和更新数据。

#### 数据收集流程

- 推理相关数据

  - 拒绝采样：在推理导向的 RL **模型 checkpoint 上**，对同⼀ prompt 采样多个回答，再从中挑选正确或质量更⾼的输出。这种⽅式可有效过滤掉错误或低质量的回答，得到⼲净的监督数据。

  > 这里值得提的一点是，这样得到的问答对数据集仍然**符合模型自身的数据分布**

  - 数据来源：除了之前使⽤过的基于规则的奖励（与 Zero 训练数据类似）的数据外，此阶段还引⼊了更多类型的数据，包括使⽤⽣成式奖励模型（generative reward model）的场景。这些场景中，需要将模型的预测结果和真实答案⼀起输⼊到 DeepSeek-V3 中来进⾏评价。 

  - 数据过滤：由于模型的输出可能过于混乱或可读性差，需要过滤掉⼀些混合语⾔、段落过⻓、包含代码块等不符合预期格式的推理链。

  - 最终规模：经过上述过程，为推理相关的训练数据共收集到约 60 万条。 

- ⾮推理相关数据

  - 数据类型：写作、事实性问答、⾃我认知、翻译等。 

  - 数据来源：沿⽤ DeepSeek-V3 的pipeline，并部分复⽤ DeepSeek-V3 的 SFT 数据。

  - CoT 的使⽤：对于某些需要进⼀步推理或分析的场景，可以调⽤ DeepSeek-V3 先⽣成⼀个潜在的推理链，然后再给出答案；但对于“hello”这样⾮常简单的对话，就不再需要提供任何推理链。 

  - 数据规模：最终收集到的⾮推理数据约 20 万条。

#### 微调阶段

将以上两种类型数据（推理 + ⾮推理），合计约 80 万条 数据，来对 **DeepSeek-V3-Base** 进⾏ 2 epoch 的监督微调。 
 
⽬标：让模型在推理⽅⾯保持优势的同时，具备多领域的通⽤能⼒，以及更好的可读性和多任务表现


<div style="page-break-after: always;"></div>

# 2026/8/14

### 3. 面向所有场景的强化学习

这是 R1 流程的第三次强化学习，在保留推理能力的基础上，进一步让模型适应更广泛的真实使用场景。

- 推理数据 (reasoning data)

  沿用第一阶段/DeepSeek-R1-Zero 中的推理任务构造方式，对数学、代码等可验证任务使用基于规则的奖励。

- 通用数据 (general data)

  引入更加多样化的通用指令数据。对于无法通过简单规则判断质量的任务，利用基于人类偏好的 Reward Model进行评分。Reward Model 本身由人工标注的偏好数据进行训练。

  > **Reward Model** 是一个经过专门训练的语言模型，**输入“问题 + 模型回答”，输出一个标量分数**，用来衡量这个回答有多符合人类偏好。

<div align="center">

<img src="note_pics/rl2-reward.png" >
<br>
该阶段强化学习奖励函数构成
</div>

该阶段的其他训练参数基本沿用第一阶段，但实验发现 temperature 设置为 1.0 时生成文本的流畅性较差，因此将采样温度降低至 0.7。

该阶段共进行 1700 steps，并在**训练后期的 400 steps 中**加入通用指令数据以及基于人类偏好的奖励信号。


## 知识蒸馏：赋予小模型推理能力

### 1. 使⽤⼤规模数据进⾏微调

为了赋予⼩模型推理能⼒，研究者使⽤了 DeepSeek-R1 微调过程中收集到的 80 万条训练数据（第三步的 SFT 过程用到的数据集），对⼀些开源⼤模型进⾏微调。

- 具体⽅法如下：

  - 使⽤的模型：研究者选择了开源模型，如 Qwen和 Llama，并使⽤这些模型进⾏微调。 

    - Qwen2.5 系列：Qwen2.5-Math-1.5B、Qwen2.5-Math-7B、Qwen2.5-14B、Qwen2.5-32B。 

    - Llama 系列：Llama-3.1-8B、Llama-3.3-70B-Instruct

    选择 Llama-3.3：由于 Llama-3.3 的推理能⼒略强于 Llama-3.1，因此在这些模型中选择了 Llama-3.3 来进⾏蒸馏。 

- 蒸馏过程的重点：仅使⽤ SFT，使用 Hard-label 蒸馏

  在蒸馏过程中，研究者仅使⽤监督微调，没有引⼊强化学习。尽管引⼊ RL 可以显著提升模型性能，但此时的主要⽬标是展示 知识蒸馏⽅法的有效性，并通过这种⽅式赋予⼩型模型推理能⼒

- 结论

  蒸馏后的模型依然能够显著提升推理性能，证明了 知识蒸馏 技术在提升⼩模型推理能⼒⽅⾯的巨⼤潜⼒。
  
### 2. 常见模型蒸馏方法

<div align="center">


| 方法                        | Teacher 给 Student 什么 | Student 学什么             |
| ------------------------- | -------------------- | ----------------------- |
| **Hard-label 蒸馏**         | Teacher 生成的 token    | 学 Teacher 最终选择          |
| **Soft-label / Logit 蒸馏** | Teacher 的概率分布        | 学 Teacher 对候选 token 的偏好 |
| **CoT 蒸馏**                | Teacher 的推理文本        | 学显式的推理过程                |
| **Soft Prompt 蒸馏**        | Teacher 的能力作为监督信号    | 学一组连续 Prompt 向量         |

</div>

软硬标签蒸馏

- 软标签蒸馏是将教师模型对每个 token 的概率分布作为软标签，与学生模型的预测分布进行损失计算，**软标签包含了教师对每个 token 的置信度信息**，因此能够提供更丰富的监督信号。

  > 教师的 softmax 分布本身并不等于教师的思考过程；如果教师显式生成了 CoT，并将其作为训练目标，才能说学生学习了教师展示出的推理过程。

- 硬标签蒸馏只使用教师生成的 token 作为硬标签，监督信号较简单

软提示蒸馏

  - 在给教师模型和学生模型输入相同 Prompt 的基础上，在学生模型的输入 embedding 前加入一组额外的可学习连续向量作为“软提示”，用于引导学生模型产生接近教师模型的输出。模型实际会看到

  $$
  [\text{Soft Prompt Embeddings}]+[\text{Original Prompt Embeddings}]
  $$

  - 在典型的 Soft Prompt 方法中，学生模型参数保持冻结，只更新 Soft Prompt 本身
  
  - 如果采用动态 Soft Prompt，则可以通过一个 Prompt Generator 根据原始 Prompt 生成 Soft Prompt，并训练该 Generator 的参数。

## 模型输出前的经历

$$
\boxed{
\text{Logits}
\rightarrow
\text{Temperature}
\rightarrow
\text{Top-K/Top-P}
\rightarrow
\text{Normalize}
\rightarrow
\text{Sampling}
}
$$

### 1. Temperature

模型最后会输出一组 token 的 logits：$z_1,z_2,\cdots,z_n$，正常情况下使用 Softmax 函数得到选取每个token的概率：

$$
P_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

Temperature 参数会把 logits 除以一个温度 $T$：

$$
P_i=\frac{e^{z_i/T}}{\sum_j e^{z_j/T}}
$$

$T$ 位于分母，相当于对于原本的概率分布进行缩放。此处只考虑 Softmax 函数的分子数值即可，下面是为了便于理解的分类讨论：

- $T<1$ 时，由于 $T$ 位于分母，所有指数项都变大；又由于是以 $e$ 为底的指数变化，因此较大的 $z_i$ 会有更大的 $e^{z_j}$ 数值。

  因此 $T<1$ 时，整个 Softmax 分布变得更加尖锐，模型输出更加会倾向于原本计划选择的 token ，模型输出更加稳定（因为温度低更加**冷静**）

- $T>1$ 时，由于 $T$ 位于分母，所有指数项都变小；又由于是以 $e$ 为底的指数变化，因此较大的 $z_i$ 会变小很多。

  因此 $T<1$ 时，整个 Softmax 分布变得更加均匀，模型输出会考虑到原本不在选择中的 token ，模型输出更加有创造力（因为温度高更加**冲动**）

$$
T<1\Rightarrow 分布更尖锐, T>1\Rightarrow 分布更平滑
$$

### 2. Top-k & Top-p 采样

- Top-k

  核心思想是**只保留概率最高的 K 个 token**，假设模型预测下一个token结果为

  <div align="center">
  
    | Token |   概率 |
  | ----- | ---: |
  | **A**    | **0.40** |
  | **B**     | **0.25** |
  | **C**     | **0.15** |
  | D     | 0.10 |
  | E     | 0.05 |
  | F     | 0.03 |
  | G     | 0.02 |
  
  </div>

- Top-p

  Top-P 又叫 Nucleus Sampling（核采样）。它是**从概率最高的 token 开始累加，直到累计概率达到 $P$ 为止**，保留的 token 数量是不固定的。
  
  假设 $P=0.8$ ，按照刚刚的例子，按照概率从高到低：

  $$
  A = 0.40, 累计 = 0.40
  $$
  $$
  B = 0.25, 累计 = 0.65
  $$
  $$
  C = 0.15, 累计 = 0.80  ← 达到 P
  $$
  
  这样模型就选取了 3 个候选的 token 
  当模型对于**输出比较确定**的时候，按照此方法选择的 token 数量会更少，如 $A = 0.90$ 则直接满足 $P$ 要求。

> 现在主流采样方式都是 Top-P ，核心原因是 Top-P 能**根据当前概率分布动态调整候选 token 数量**。值得注意的是，现在很多推理模型（例如 DeepSeek-R1 这一类）在实际使用时，并不一定鼓励用户通过很高的 temperature、Top-P 等参数来追求随机性。因为推理任务更关心**稳定、可靠地得到正确答案**，而不是生成尽可能多样的文本。

### 3. 归一化后随机选取

经过上一步的选择已经获得了若干个候选 token，那么如何选择具体的 **那一个token** 作为输出？

- 归一化

  按照上面的例子继续考虑，如果 $K=3$，那么只保留概率最高的 3 个，即 Token A、B、C，然后把概率重新归一化
  
    $$
    P'(x)=\frac{P(x)}{0.40+0.25+0.15}
    $$
  
  假设这三个 token 分别代表三种具体的动物名称，带入数值有
  
    $$
    P(\text{猫})=\frac{0.40}{0.80}=0.5
    $$
    $$
    P(\text{狗})=\frac{0.25}{0.80}=0.3125
    $$
    $$
    P(\text{鸟})=\frac{0.15}{0.80}=0.1875
    $$

- 随机采样

  从0-1区间找随机数，看落在哪个区间就选择哪个 token 作为本次预测的输出

